# ViFinQA clean canonical baseline v1
Schema-9, Selection v2 only, no ID masks, no public-derived gold. Default model is Qwen2.5-Coder-7B-Instruct-AWQ; the organizer-confirmed limit is 15B.

In [ ]:
import json
from pathlib import Path

manifests = []
for path in Path('/kaggle/input').rglob('payload-manifest.json'):
    obj = json.loads(path.read_text())
    if obj.get('schema_version') == 9 and obj.get('profile') == 'clean':
        manifests.append(path)
assert len(manifests) == 1, f'expected one clean schema-9 payload, got {manifests}'
PAYLOAD = manifests[0].parent
RUNNER = PAYLOAD / 'code' / 'kaggle_clean_codegen.py'
print(PAYLOAD)

In [ ]:
import subprocess

cmd = [
    'python', str(RUNNER), '--payload', str(PAYLOAD),
    '--out', '/kaggle/working/codegen_results.jsonl',
    '--backend', 'hf',
    '--model', 'Qwen/Qwen2.5-Coder-7B-Instruct-AWQ',
    '--llm-mode', 'select_v2', '--llm-target', 'all',
    '--k', '0', '--n', '2', '--temperature', '0.2',
    '--max-tokens', '512', '--seed', '13',
]
subprocess.run(cmd, check=True)

In [ ]:
import sys
sys.path.insert(0, str(PAYLOAD / 'code'))
from vifinqa.submission.build import build_submission

zip_path = build_submission(
    PAYLOAD / 'retrieval.jsonl',
    Path('/kaggle/working/codegen_results.jsonl'),
    PAYLOAD / 'store',
    Path('/kaggle/working/submission'),
    sub_k=5, pos_mode='line', expand_docs=False,
)
print(zip_path)